# Module 11 lab · Pretraining for real

The code from every lesson of [this module](https://aicourses.dev/research/pretraining), in order. Read a lesson, then run its cells here: each lesson builds on the one before, so run the cells from the top the first time.

**This lab needs a GPU.** Colab opens it on one; if not, choose Runtime → Change runtime type → T4 GPU.

**First time?** Read the [one-time setup](https://aicourses.dev/research/labs): a Google account, your own copy, and what to expect.

The lessons that explain this code are part of [AI Courses Pro](https://aicourses.dev/#pricing); the module’s first lesson is free.

Under each cell, *The lesson shows* is what it should print. Your last digits can differ a little.

**Keep your work:** File → Save a copy in Drive. Colab does not keep changes to this copy.

## Setup: 10.2 Self-attention, one head, step by step

This module carries on from [lesson 10.2](https://aicourses.dev/research/transformers/self-attention). Run these cells first.

In [ ]:
import torch
import torch.nn.functional as F
from torch import nn

words = ["Mary", "visited", "Paris", "she"]
K = torch.tensor([[1.0, 0.0],    # Mary: a person
                  [0.0, 0.0],    # visited: neither
                  [0.0, 1.0],    # Paris: a place
                  [0.5, 0.0]])   # she: a little person-like
q_she = torch.tensor([2.0, 0.0]) # she asks: "who is a person?"
scores = K @ q_she               # one score per word
weights = F.softmax(scores, dim=0)
for w, s, a in zip(words, scores.tolist(), weights.tolist()):
    print(f"{w:8} score {s:.1f}  weight {a:.3f}")
# Mary     score 2.0  weight 0.610
# visited  score 0.0  weight 0.083
# Paris    score 0.0  weight 0.083
# she      score 1.0  weight 0.225

In [ ]:
torch.manual_seed(0)
for d in [4, 64, 512]:
    q, k = torch.randn(10000, d), torch.randn(10000, d)
    dots = (q * k).sum(dim=1)
    print(d, round(dots.std().item(), 2),
          round((dots / d**0.5).std().item(), 2))
# 4 1.99 1.0
# 64 8.01 1.0
# 512 22.49 0.99

In [ ]:
torch.manual_seed(42)
T, C, hs = 4, 8, 4       # 4 tokens, 8 numbers each, head size 4
x = torch.randn(T, C)    # one embedding per token
Wq = torch.randn(C, hs) / C**0.5   # learned in a real model
Wk = torch.randn(C, hs) / C**0.5
Wv = torch.randn(C, hs) / C**0.5
q, k, v = x @ Wq, x @ Wk, x @ Wv   # each (4, 4)
scores = q @ k.T / hs**0.5         # (4, 4): every query, every key
weights = F.softmax(scores, dim=-1)  # each row adds up to 1
out = weights @ v                  # (4, 4): one blend per token
print(weights.round(decimals=2))
print(out.shape)                   # torch.Size([4, 4])

## Setup: 10.3 Causal masking and why GPT can’t peek

This module carries on from [lesson 10.3](https://aicourses.dev/research/transformers/causal-masking). Run these cells first.

In [ ]:
T = 4
mask = torch.tril(torch.ones(T, T, dtype=torch.bool))
print(mask.int())
scores = torch.zeros(T, T)          # pretend every score is 0
scores = scores.masked_fill(~mask, float("-inf"))
print(F.softmax(scores, dim=-1).round(decimals=2))

In [ ]:
def head(x, Wq, Wk, Wv, causal=True):
    q, k, v = x @ Wq, x @ Wk, x @ Wv
    scores = q @ k.T / k.shape[-1] ** 0.5
    if causal:                      # hide every later token
        T = x.shape[0]
        mask = torch.tril(torch.ones(T, T, dtype=torch.bool))
        scores = scores.masked_fill(~mask, float("-inf"))
    return F.softmax(scores, dim=-1) @ v

In [ ]:
torch.manual_seed(42)
x = torch.randn(4, 8)
Wq, Wk, Wv = [torch.randn(8, 4) / 8 ** 0.5 for _ in range(3)]
x2 = x.clone()
x2[3] = torch.randn(8)              # change only the last token
for causal in [True, False]:
    a = head(x, Wq, Wk, Wv, causal)
    b = head(x2, Wq, Wk, Wv, causal)
    print(causal, (a[:3] - b[:3]).abs().max().item())
# True 0.0
# False 0.520298421382904

## Setup: 10.4 Multi-head attention and projections

This module carries on from [lesson 10.4](https://aicourses.dev/research/transformers/multi-head-attention). Run these cells first.

In [ ]:
torch.manual_seed(42)
B, T, C, nh = 2, 5, 16, 4        # batch, tokens, channels, heads
hs = C // nh                     # head size: 4
x = torch.randn(B, T, C)
qkv = nn.Linear(C, 3 * C)        # all three projections in one
q, k, v = qkv(x).split(C, dim=2)            # each (2, 5, 16)
q = q.view(B, T, nh, hs).transpose(1, 2)    # (2, 4, 5, 4)
k = k.view(B, T, nh, hs).transpose(1, 2)
v = v.view(B, T, nh, hs).transpose(1, 2)
att = q @ k.transpose(-2, -1) / hs ** 0.5   # (2, 4, 5, 5)
mask = torch.tril(torch.ones(T, T, dtype=torch.bool))
att = att.masked_fill(~mask, float("-inf")).softmax(dim=-1)
y = att @ v                                 # (2, 4, 5, 4)
y = y.transpose(1, 2).reshape(B, T, C)      # (2, 5, 16)
print(q.shape, att.shape, y.shape)

In [ ]:
ref = F.scaled_dot_product_attention(q, k, v, is_causal=True)
ref = ref.transpose(1, 2).reshape(B, T, C)
print((y - ref).abs().max().item())   # 8.940696716308594e-08

In [ ]:
q2, k2, v2 = qkv(x).split(C, dim=2)
wrong = q2.reshape(B, nh, T, hs)            # skips the transpose
print((wrong - q).abs().max().item())       # 3.0071001052856445

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.n_head = n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd)
        self.proj = nn.Linear(n_embd, n_embd)   # mixes the heads

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        split = lambda t: t.view(B, T, self.n_head, -1).transpose(1, 2)
        y = F.scaled_dot_product_attention(
            split(q), split(k), split(v), is_causal=True)
        y = y.transpose(1, 2).reshape(B, T, C)  # join the heads
        return self.proj(y)

attn = CausalSelfAttention(768, 12)          # GPT-2 small's sizes
print(sum(p.numel() for p in attn.parameters()))   # 2362368

## Setup: 10.5 Positional information: from learned positions to RoPE

This module carries on from [lesson 10.5](https://aicourses.dev/research/transformers/positions-and-rope). Run these cells first.

In [ ]:
torch.manual_seed(42)
x = torch.randn(3, 8)            # "dog", "bites", "man"
Wq, Wk, Wv = [torch.randn(8, 4) / 8 ** 0.5 for _ in range(3)]
order = [2, 1, 0]                # "man", "bites", "dog"
a = head(x, Wq, Wk, Wv, causal=False)
b = head(x[order], Wq, Wk, Wv, causal=False)
print((a[order] - b).abs().max().item())   # 2.384185791015625e-07

In [ ]:
vocab_size, n_embd, block_size = 65, 32, 64   # small example sizes
idx = torch.randint(vocab_size, (2, 8))         # 2 texts of 8 tokens
B, T = idx.shape
wte = nn.Embedding(vocab_size, n_embd)   # one row per token
wpe = nn.Embedding(block_size, n_embd)   # one row per position
x = wte(idx) + wpe(torch.arange(T))      # (B, T, n_embd)

In [ ]:
def rope(x, pos, base=10000.0):
    d = x.shape[-1]                       # must be even
    freqs = base ** (-torch.arange(0, d, 2) / d)  # one per pair
    ang = pos[:, None] * freqs[None, :]   # (T, d/2) angles
    cos, sin = ang.cos(), ang.sin()
    x1, x2 = x[..., 0::2], x[..., 1::2]   # the two halves of pairs
    turned = torch.stack([x1 * cos - x2 * sin,
                          x1 * sin + x2 * cos], dim=-1)
    return turned.flatten(-2)

In [ ]:
torch.manual_seed(0)
q, k = torch.randn(1, 64), torch.randn(1, 64)
for m, n in [(5, 3), (10, 8), (1005, 1003), (5, 4)]:
    qm = rope(q, torch.tensor([float(m)]))
    kn = rope(k, torch.tensor([float(n)]))
    print(m, n, round((qm @ kn.T).item(), 4))
# 5 3 -10.1427
# 10 8 -10.1427
# 1005 1003 -10.1427
# 5 4 -11.4563

## Setup: 10.6 The transformer block: residuals, layer norm, MLP

This module carries on from [lesson 10.6](https://aicourses.dev/research/transformers/transformer-block). Run these cells first.

In [ ]:
def grads(residual, depth=30):
    torch.manual_seed(42)
    layers = [nn.Linear(32, 32) for _ in range(depth)]
    x = torch.randn(64, 32)
    for layer in layers:
        out = torch.tanh(layer(x))
        x = x + out if residual else out   # the shortcut, or not
    x.pow(2).mean().backward()
    first = layers[0].weight.grad.norm().item()
    last = layers[-1].weight.grad.norm().item()
    return f"first {first:.1e}  last {last:.1e}"

print("plain:   ", grads(residual=False))
print("residual:", grads(residual=True))
# plain:    first 7.0e-10  last 3.0e-02
# residual: first 6.3e+00  last 4.5e+00

In [ ]:
ln = nn.LayerNorm(4)
print(ln(torch.tensor([1.0, 2.0, 3.0, 4.0])).detach())
# tensor([-1.3416, -0.4472,  0.4472,  1.3416])

In [ ]:
class MLP(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.fc = nn.Linear(n_embd, 4 * n_embd)     # widen
        self.proj = nn.Linear(4 * n_embd, n_embd)   # narrow back

    def forward(self, x):
        return self.proj(F.gelu(self.fc(x)))

In [ ]:
class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = MLP(n_embd)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))   # read, then add back
        x = x + self.mlp(self.ln2(x))
        return x

block = Block(768, 12)                   # GPT-2 small's sizes
print(sum(p.numel() for p in block.parameters()))   # 7087872

## Setup: 10.7 Training GPT on Tiny Shakespeare

This module carries on from [lesson 10.7](https://aicourses.dev/research/transformers/training-gpt). Run these cells first.

In [ ]:
import math
import urllib.request

url = ("https://raw.githubusercontent.com/karpathy/char-rnn/"
       "master/data/tinyshakespeare/input.txt")
urllib.request.urlretrieve(url, "input.txt")
text = open("input.txt").read()
chars = sorted(set(text))                 # every symbol used
stoi = {ch: i for i, ch in enumerate(chars)}
data = torch.tensor([stoi[ch] for ch in text])
n = int(0.9 * len(data))
train, val = data[:n], data[n:]           # first 90% to train
print(len(text), len(chars))              # 1115394 65

In [ ]:
block_size, batch_size = 64, 32
gen = torch.Generator().manual_seed(42)   # for training batches

def get_batch(split, g=gen):
    d = train if split == "train" else val
    ix = torch.randint(len(d) - block_size, (batch_size,),
                       generator=g)
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + 1 + block_size] for i in ix])
    return x, y                           # y is x shifted by one

In [ ]:
class GPT(nn.Module):
    def __init__(self, vocab, block_size, n_layer, n_head, n_embd):
        super().__init__()
        self.wte = nn.Embedding(vocab, n_embd)       # tokens
        self.wpe = nn.Embedding(block_size, n_embd)  # positions
        self.blocks = nn.Sequential(
            *[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab, bias=False)
        self.apply(self._init)

    def _init(self, m):                  # GPT-2's starting weights
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, idx):
        T = idx.shape[1]
        x = self.wte(idx) + self.wpe(torch.arange(T))  # (B, T, C)
        return self.head(self.ln_f(self.blocks(x)))    # (B, T, V)

In [ ]:
def loss_on(model, x, y):
    logits = model(x)
    return F.cross_entropy(logits.view(-1, logits.size(-1)),
                           y.view(-1))

torch.manual_seed(42)
model = GPT(len(chars), block_size, n_layer=4, n_head=4, n_embd=128)
print(sum(p.numel() for p in model.parameters()))   # 818176
x, y = get_batch("train")
print(round(loss_on(model, x, y).item(), 3),
      round(math.log(len(chars)), 3))               # 4.138 4.174

In [ ]:
x2 = x.clone()
x2[:, -1] = (x2[:, -1] + 1) % len(chars)  # change the last token
with torch.no_grad():
    diff = (model(x)[:, :-1] - model(x2)[:, :-1]).abs().max()
print(diff.item())                        # 0.0

In [ ]:
torch.manual_seed(0)
tiny = GPT(len(chars), block_size, n_layer=4, n_head=4, n_embd=128)
opt = torch.optim.AdamW(tiny.parameters(), lr=1e-3)
for step in range(200):                   # one batch, over and over
    loss = loss_on(tiny, x, y)
    opt.zero_grad()
    loss.backward()
    opt.step()
print(round(loss.item(), 3))              # 0.019

In [ ]:
egen = torch.Generator().manual_seed(1)   # for evaluation batches

@torch.no_grad()
def evaluate(split, batches=50):
    model.eval()
    total = sum(loss_on(model, *get_batch(split, egen)).item()
                for _ in range(batches))
    model.train()
    return total / batches

opt = torch.optim.AdamW(model.parameters(), lr=1e-3,
                        weight_decay=0.1)
for step in range(3001):
    if step % 500 == 0:
        print(step, round(evaluate("train"), 3),
              round(evaluate("val"), 3))
    if step == 3000:
        break
    x, y = get_batch("train")
    loss = loss_on(model, x, y)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()

In [ ]:
@torch.no_grad()
def generate(idx, n, temperature=1.0, generator=None):
    for _ in range(n):
        logits = model(idx[:, -block_size:])[:, -1] / temperature
        probs = F.softmax(logits, dim=-1)
        nxt = torch.multinomial(probs, 1, generator=generator)
        idx = torch.cat([idx, nxt], dim=1)
    return idx

g = torch.Generator().manual_seed(42)
start = torch.zeros((1, 1), dtype=torch.long)   # a newline
out = generate(start, 300, generator=g)
print("".join(chars[i] for i in out[0].tolist()))

*The lesson shows:*

```text
Mistrous shadows, and not the nost to tnough countery
Bear heed lamp on, the crave to our pland hell
unto this u
ceavier famen, for the noble leadiness?

Lady Must Derby:
Plead here, though he hast? dost he should should
```

## 11.2 Resource accounting: parameters, FLOPs and memory before you launch

Training a model with N parameters on D tokens costs about 6 × N × D FLOPs, and each parameter needs a few bytes of memory. These two sums tell you whether a run fits your free GPU before you spend hours on it.

[Read the lesson](https://aicourses.dev/research/pretraining/resource-accounting)

### Counting parameters

You counted parameters for one block in lesson 10.6. Here is all of GPT-2 small, as a formula. With $C$ numbers per token, a block holds $12C^2$ weights: $4C^2$ in attention (four C × C matrices) and $8C^2$ in the MLP (C × 4C and 4C × C). It also holds $13C$ biases and layer-norm numbers. At $C = 768$ that is 7,077,888 + 9,984 = 7,087,872, lesson 10.6’s count. Add the token table, the position table and the final layer norm:

In [ ]:
def gpt_params(vocab, ctx, n_layer, C):
    embed = vocab * C + ctx * C        # token and position tables
    per_block = 12 * C * C + 13 * C    # weights, biases, 2 norms
    return embed + n_layer * per_block + 2 * C   # + final norm

print(gpt_params(50257, 1024, 12, 768))   # 124439808

Check it against the real thing, the GPT class from lesson 10.7 at GPT-2 small’s sizes. This module’s code carries on in your Module 10 notebook, which holds lesson 10.7’s GPT class, the Shakespeare text and your trained model: open it, or in a new runtime run Module 10’s cells again first. GPT-2 uses one matrix for both the token table and the output layer, a trick called **weight tying**, so they are counted once:

In [ ]:
gpt2 = GPT(50257, 1024, n_layer=12, n_head=12, n_embd=768)
gpt2.head.weight = gpt2.wte.weight     # GPT-2 shares this matrix
print(sum(p.numel() for p in gpt2.parameters()))    # 124439808

### Arithmetic: 6 × N × D

Measure it. PyTorch can count the FLOPs of everything it runs:

In [ ]:
from torch.utils.flop_counter import FlopCounterMode
from torch.nn.attention import sdpa_kernel, SDPBackend

torch.manual_seed(0)
small = GPT(65, 64, n_layer=4, n_head=4, n_embd=128)
N = sum(p.numel() for p in small.parameters())
x = torch.randint(0, 65, (32, 64))     # one batch: 2,048 tokens
counter = FlopCounterMode(display=False)
with sdpa_kernel(SDPBackend.MATH), counter:
    small(x).sum().backward()          # forward and backward
measured = counter.get_total_flops()
print(N, measured, 6 * N * x.numel())
# 818176 10571218944 10053746688

### From FLOPs to hours

The time is the arithmetic divided by the speed you actually reach:

In [ ]:
def plan(N, D, peak_tflops, mfu=0.4):
    flops = 6 * N * D                   # training arithmetic
    memory_gb = 16 * N / 1e9            # weights, grads, Adam
    seconds = flops / (peak_tflops * 1e12 * mfu)
    return flops, memory_gb, seconds / 3600

N = 124_439_808                         # GPT-2 small
for gpu, peak in [("T4", 65), ("A100", 312), ("H100", 989)]:
    flops, mem, hours = plan(N, 10e9, peak)
    print(f"{gpu:5} {flops:.2e} FLOPs {mem:.1f} GB {hours:6.1f} h")
# T4    7.47e+18 FLOPs 2.0 GB   79.8 h
# A100  7.47e+18 FLOPs 2.0 GB   16.6 h
# H100  7.47e+18 FLOPs 2.0 GB    5.2 h

## 11.3 How computers store numbers: bits, floats and precision

A computer stores each number in a fixed number of bits, which limits how big and how exact it can be. Fewer bits mean faster training and less memory, and knowing what is lost makes the next lesson make sense.

[Read the lesson](https://aicourses.dev/research/pretraining/how-computers-store-numbers)

### Bits and whole numbers

A **bit** is a 0 or a 1. Eight bits make a **byte**. Whole numbers are written in base 2, so 5 is `101` (one 4, no 2, one 1) and 200 is `11001000` (128 + 64 + 8). With 8 bits there are $2^8 = 256$ patterns, so an 8-bit whole number can be 0 to 255, or, if it needs a sign, −128 to 127.

In [ ]:
print(bin(5), bin(200), int("11001000", 2))   # 0b101 0b11001000 200
print(torch.iinfo(torch.int8).min, torch.iinfo(torch.int8).max)
# -128 127

### Floats: digits and a moving decimal point

**BF16**, short for brain floating point, was created at Google Brain for Google’s TPU chips, its own AI processors. It keeps FP32’s 8 exponent bits, so it has the same range, and gives up fraction bits. FP16 makes the opposite choice. FP8’s label E4M3 just names its split: 4 exponent bits and 3 mantissa bits. PyTorch reports what each format can hold:

In [ ]:
for dt in [torch.float32, torch.float16, torch.bfloat16]:
    f = torch.finfo(dt)
    print(dt, f.bits, f.max, f.eps)
# torch.float32 32 3.4028234663852886e+38 1.1920928955078125e-07
# torch.float16 16 65504.0 0.0009765625
# torch.bfloat16 16 3.3895313892515355e+38 0.0078125

### What gets lost

Three things can go wrong when a number is squeezed into fewer bits.

In [ ]:
x = 70000.0
for dt in [torch.float32, torch.float16, torch.bfloat16]:
    print(dt, torch.tensor(x, dtype=dt).item(),
          torch.tensor(1 / 3, dtype=dt).item())
# torch.float32 70000.0 0.3333333432674408
# torch.float16 inf 0.333251953125
# torch.bfloat16 70144.0 0.333984375

**Underflow.** Very small numbers can vanish. In FP16, $10^{-8}$ becomes exactly 0. In BF16 it survives, because of its wider range:

In [ ]:
print(torch.tensor(1e-8, dtype=torch.float16).item(),
      torch.tensor(1e-8, dtype=torch.bfloat16).item())
# 0.0 1.0011717677116394e-08

### Why training cares

A training step changes each weight by a tiny amount: the learning rate times the gradient, often a thousandth of the weight or less. In BF16, that change can disappear completely:

In [ ]:
w = torch.tensor(1.0, dtype=torch.bfloat16)
print((w + 0.001).item(), (w.float() + 0.001).item())
# 1.0 1.0010000467300415

## 11.4 Making training fast on one GPU: BF16, torch.compile, FlashAttention

Three tricks make GPT-2-sized runs affordable on one GPU. You store numbers in 16 bits (BF16), let PyTorch merge steps for you (torch.compile), and use FlashAttention, a faster way to compute attention.

[Read the lesson](https://aicourses.dev/research/pretraining/fast-training)

### Measure first

where $N$ is the number of parameters and $6N$ is lesson 11.2’s cost per token. Say a 124-million-parameter model runs at 10,000 tokens a second on a T4: $6 \times 124 \times 10^6 \times 10{,}000 \approx 7.4 \times 10^{12}$ FLOPs a second, out of 65 trillion, an MFU of about 11%. GPU work runs in the background of your Python code, so call `torch.cuda.synchronize()` before reading the clock, or you time how fast Python queues work, not how fast the GPU does it.

In [ ]:
import time

def tokens_per_second(step_fn, x, y, steps=50):
    for _ in range(5):
        step_fn(x, y)                    # warm up, and compile
    torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(steps):
        step_fn(x, y)
    torch.cuda.synchronize()             # wait for the GPU to finish
    return steps * x.numel() / (time.perf_counter() - start)

### 1. Sixteen-bit arithmetic with autocast

Lesson 11.3 showed what 16-bit numbers lose and why they are worth it. PyTorch’s **autocast** applies them for you: inside it, matrix multiplications run in 16 bits, while numerically delicate operations such as softmax and the loss stay in 32 bits, and the weights themselves remain 32-bit master copies. You can see it work even on Colab’s CPU runtime:

In [ ]:
x, y = get_batch("train")               # a batch, as in lesson 10.7
with torch.autocast(device_type="cpu", dtype=torch.bfloat16):
    logits = model(x)
print(logits.dtype, model.head.weight.dtype)
# torch.bfloat16 torch.float32

The output came out in BF16; the weights stayed FP32. Which 16-bit format to use depends on your GPU. NVIDIA’s A100 and H100 run BF16 on their tensor cores. The T4, the GPU a free Colab runtime often gives you, lists FP16 at 65 trillion FLOPs a second on its datasheet, and no BF16 at all. With FP16, add a **gradient scaler**, the loss scaling from lesson 11.3, so that tiny gradients do not underflow to zero.

In [ ]:
major, _ = torch.cuda.get_device_capability()
use_bf16 = major >= 8                   # A100, H100 and newer
dtype = torch.bfloat16 if use_bf16 else torch.float16
scaler = torch.amp.GradScaler(enabled=not use_bf16)
torch.set_float32_matmul_precision("high")   # TF32 where it exists

def train_step(x, y):
    with torch.autocast(device_type="cuda", dtype=dtype):
        loss = loss_on(model, x, y)
    scaler.scale(loss).backward()        # scaled only for FP16
    scaler.unscale_(opt)                 # so clipping sees real sizes
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(opt)
    scaler.update()
    opt.zero_grad(set_to_none=True)

### 2. Fewer trips to memory with torch.compile

Ordinary PyTorch runs each operation separately: a layer norm reads the activations from memory and writes them back, the next operation reads them again, and so on. **torch.compile**, added with PyTorch 2.0 in 2023, records the operations your model runs and **fuses** chains of small operations into fewer GPU programs, called kernels, so each number is fetched once and used several times. It is one line:

In [ ]:
model = torch.compile(model)

### 3. Attention without the giant table: FlashAttention

Ordinary attention builds the full table of scores, one per pair of positions, writes it to the GPU’s main memory, reads it back for the softmax, writes it again and reads it again to multiply by the values. For GPT-2 small with a batch of 16, 12 heads and 1,024 tokens, that table is large:

In [ ]:
B, heads, T, layers = 16, 12, 1024, 12
per_layer = B * heads * T * T * 2         # BF16: 2 bytes each
print(per_layer, round(per_layer * layers / 1e9, 1))
# 402653184 4.8

## 11.5 Pretraining data: sources, filtering, deduplication, quality classifiers

What a model reads decides what it knows. You see where pretraining text comes from and how teams clean it: dropping junk, removing repeats and keeping the best pages.

[Read the lesson](https://aicourses.dev/research/pretraining/pretraining-data)

### The cleaning steps

Here are toy versions of the first checks. Exact duplicates are easy: normalise the text and compare fingerprints.

In [ ]:
import hashlib
docs = [
    "The heart pumps blood around the body. It beats about "
    "once a second.",
    "the heart pumps  blood around the body. It beats about "
    "once a second.",
    "Home | About | Contact | Login | Cart",
    "BUY NOW cheap pills click here click here click here",
    "Plants turn sunlight into food in their leaves. "
    "Chlorophyll is what makes the leaves green.",
    "Plants turn sunlight into food in their leaves. "
    "Chlorophyll is what makes leaves look green.",
]

def fingerprint(text):
    normal = " ".join(text.lower().split())   # case, spaces
    return hashlib.md5(normal.encode()).hexdigest()

seen, unique = set(), []
for d in docs:
    f = fingerprint(d)
    if f not in seen:
        seen.add(f)
        unique.append(d)
print(len(docs), len(unique))                 # 6 5

`hashlib.md5` turns any text into a short code, a **hash**, that changes completely if one character changes, so equal codes mean equal texts. The second page differs only in a capital letter and a double space, so it is caught. Now some quality rules, loosely modelled on the published ones, with toy thresholds for toy pages:

In [ ]:
def keep(text):
    words = text.split()
    alpha = sum(any(c.isalpha() for c in w) for w in words)
    repeats = 1 - len(set(words)) / len(words)
    return (len(words) >= 8                       # long enough
            and text.rstrip()[-1] in ".!?\"'"     # ends a sentence
            and alpha / len(words) >= 0.8         # mostly words
            and repeats < 0.3)                    # not repetitive

for d in unique:
    print(keep(d), d[:40])

### Near-duplicates with MinHash

Comparing every pair of pages among billions is impossible. **MinHash** gives each page a short signature instead. For each of many hash functions, which turn any text into a big, random-looking number, record the smallest number any of the page’s shingles produces. Two pages get the same smallest number with a probability equal to their Jaccard similarity, so the share of matching signature entries estimates it.

In [ ]:
def shingles(text, n=3):
    w = text.lower().split()
    return {" ".join(w[i:i + n]) for i in range(len(w) - n + 1)}

def minhash(text, k=64):
    sig = []
    for seed in range(k):                     # k hash functions
        sig.append(min(int(hashlib.md5(f"{seed}{s}".encode())
                           .hexdigest(), 16) for s in shingles(text)))
    return sig

a, b = unique[-2], unique[-1]
sa, sb = shingles(a), shingles(b)
exact = len(sa & sb) / len(sa | sb)
ma, mb = minhash(a), minhash(b)
estimate = sum(x == y for x, y in zip(ma, mb)) / len(ma)
print(round(exact, 3), round(estimate, 3))    # 0.625 0.641

### Quality classifiers

The classifier is public. Give it two short pages:

In [ ]:
photosynthesis_page = (
    "Photosynthesis is the process plants use to turn light into "
    "chemical energy. Chlorophyll in the leaves absorbs sunlight, "
    "and the plant uses that energy to make sugar from water and "
    "carbon dioxide, releasing oxygen as a by-product.")
shop_page = ("Free shipping on all orders over $50! Shop our summer "
             "sale now. Sign up for our newsletter and get 10% off "
             "your first order.")

This downloads the classifier, about 440 MB, the first time:

In [ ]:
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification

name = "HuggingFaceFW/fineweb-edu-classifier"
tokenizer = AutoTokenizer.from_pretrained(name)
scorer = AutoModelForSequenceClassification.from_pretrained(name)
for page in [photosynthesis_page, shop_page]:
    inputs = tokenizer(page, return_tensors="pt", truncation=True)
    score = scorer(**inputs).logits.squeeze(-1).item()
    print(round(score, 2), int(round(max(0, min(score, 5)))))
# 3.1 3
# -0.15 0

## 11.6 Reproducing GPT-2 small: shards, the run, and babysitting a loss curve

Puts the last three lessons together in one real run, with checkpoint-and-resume and triage of loss spikes.

[Read the lesson](https://aicourses.dev/research/pretraining/reproducing-gpt-2)

### The data, in shards

Shards let the run read data in order, from disk, without ever holding all of it in memory, and they make “where was I?” a simple question: which shard, and how far in. Here is the same idea on Tiny Shakespeare’s training characters, lesson 10.7’s `train`, in shards of 100,000:

In [ ]:
import glob
import numpy as np

def write_shards(tokens, shard_size, prefix):
    for i in range(0, len(tokens), shard_size):
        part = np.array(tokens[i:i + shard_size], dtype=np.uint16)
        np.save(f"{prefix}_{i // shard_size:06d}.npy", part)

write_shards(train, 100_000, "shakespeare_train")
files = sorted(glob.glob("shakespeare_train_*.npy"))
print(len(files), np.load(files[0]).dtype, np.load(files[-1]).shape)
# 11 uint16 (3854,)

A loader walks through the shards in order and can report, and jump back to, its exact position:

In [ ]:
class ShardLoader:
    def __init__(self, files, B, T):
        self.files, self.B, self.T = files, B, T
        self.reset(0, 0)

    def reset(self, shard, pos):          # jump to any place
        self.shard, self.pos = shard, pos
        arr = np.load(self.files[shard]).astype(np.int64)
        self.tokens = torch.from_numpy(arr)

    def next_batch(self):
        n = self.B * self.T
        buf = self.tokens[self.pos:self.pos + n + 1]
        x = buf[:-1].view(self.B, self.T)     # inputs
        y = buf[1:].view(self.B, self.T)      # next tokens
        self.pos += n
        if self.pos + n + 1 > len(self.tokens):   # on to next shard
            self.reset((self.shard + 1) % len(self.files), 0)
        return x, y

    def state(self):
        return {"shard": self.shard, "pos": self.pos}

### The run’s settings

In [ ]:
import math

def get_lr(step, max_lr=6e-4, warmup=715, max_steps=19073):
    min_lr = max_lr * 0.1
    if step < warmup:                       # straight line up
        return max_lr * (step + 1) / warmup
    if step > max_steps:
        return min_lr
    ratio = (step - warmup) / (max_steps - warmup)
    coeff = 0.5 * (1 + math.cos(math.pi * ratio))   # 1 down to 0
    return min_lr + coeff * (max_lr - min_lr)

for s in [0, 357, 714, 5000, 10000, 19073]:
    print(s, f"{get_lr(s):.2e}")
# 0 8.39e-07
# 357 3.00e-04
# 714 6.00e-04
# 5000 5.31e-04
# 10000 3.25e-04
# 19073 6.00e-05

### Checkpoints that resume exactly

Free Colab sessions end, machines fail, and big runs are interrupted all the time. A checkpoint must hold everything the next step depends on: the weights, the optimizer’s state, the position in the data, the random-number state and the step number.

In [ ]:
def save(path, model, opt, loader, step):
    torch.save({"model": model.state_dict(),
                "opt": opt.state_dict(),     # AdamW's averages too
                "data": loader.state(),      # where we were reading
                "rng": torch.get_rng_state(),
                "step": step}, path)

def load(path, model, opt, loader):
    ck = torch.load(path)
    model.load_state_dict(ck["model"])
    opt.load_state_dict(ck["opt"])
    loader.reset(**ck["data"])
    torch.set_rng_state(ck["rng"])
    return ck["step"]

In Colab, give `save` a path inside your mounted Google Drive (lesson 2.8), because the runtime’s own disk goes when the session ends; in a new session you mount Drive again, `load` the latest checkpoint and carry on. `torch.get_rng_state` saves where the random-number generator has got to, so a resumed run draws the same random numbers, and `**ck["data"]` passes the saved shard and position back in as named arguments. Test it the way you test anything that must be exact. Two short helpers come first: `make`, which builds a fresh small GPT, its optimizer and the data loader from a fixed seed, and `train_steps`, your training loop from lesson 10.7, made to run a set number of steps and return the list of losses. Then:

In [ ]:
def make():                         # a small GPT, its optimizer, a loader at the start
    torch.manual_seed(0)
    model = GPT(65, 64, n_layer=2, n_head=4, n_embd=64)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
    return model, opt, ShardLoader(files, 8, 64)

def train_steps(model, opt, loader, steps):   # returns each step's loss
    losses = []
    for _ in range(steps):
        x, y = loader.next_batch()
        loss = loss_on(model, x, y)
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
    return losses

straight = train_steps(*make(), steps=200)

net, opt, loader = make()
first = train_steps(net, opt, loader, steps=100)
save("ckpt.pt", net, opt, loader, step=100)
del net, opt, loader                     # the machine "crashes"

net, opt, loader = make()                # a fresh start...
step = load("ckpt.pt", net, opt, loader)  # ...then resume
second = train_steps(net, opt, loader, steps=100)
print(step, straight == first + second)    # 100 True

## 11.7 Scaling laws: fitting how loss falls with compute

Bigger models trained on more data reach lower loss, in a pattern you can draw as a straight line. Fitting that line on tiny models predicts big ones, and explains Chinchilla’s rule for how much data a model of each size needs.

[Read the lesson](https://aicourses.dev/research/pretraining/scaling-laws)

### Fit one yourself

Train the Shakespeare GPT of lesson 10.7 at five widths, 16, 32, 64, 128 and 256 numbers per token, each with 4 layers, for the same 1,500 steps, so each reads the same 3,072,000 characters. `get_batch` and `loss_on` are as in lesson 10.7, with the batch generator passed in:

In [ ]:
def run(n_embd, steps=1500):
    torch.manual_seed(42)
    model = GPT(65, block_size, n_layer=4, n_head=4, n_embd=n_embd)
    N = sum(p.numel() for p in model.parameters())
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
    gen = torch.Generator().manual_seed(42)
    for _ in range(steps):
        loss = loss_on(model, *get_batch(train, gen))
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
    egen = torch.Generator().manual_seed(1)
    with torch.no_grad():
        v = sum(loss_on(model, *get_batch(val, egen)).item()
                for _ in range(50)) / 50
    return N, v

Hold back the biggest one, fit a straight line to the logs of the other four, and predict it:

In [ ]:
import numpy as np

N = np.array([16256, 57088, 212480, 818176])     # parameters
L = np.array([2.2006, 2.0391, 1.908, 1.7719])    # validation loss
slope, intercept = np.polyfit(np.log(N), np.log(L), 1)
predict = lambda n: np.exp(intercept) * n ** slope
print(round(slope, 4), round(predict(3209216), 3))   # -0.0548 1.643

## 11.8 Evaluating a base model: loss, bits per byte and benchmarks

A benchmark is a fixed set of test questions that many models are scored on. Bits per byte is the loss, measured so that models with different tokenizers compare fairly. Together they show whether your run worked, and they raise Module 12’s question: how sure can you be?

[Read the lesson](https://aicourses.dev/research/pretraining/evaluating-base-models)

### Bits per byte

For your model: a loss of 1.622 nats per character, one byte each, gives $1.622 / 0.693 \approx 2.34$ bits per byte. For GPT-2 small: 4.00 nats per token over 36,059 tokens, spread over 111,540 bytes, gives $4.00 \times 36{,}059 / (0.693 \times 111{,}540) \approx 1.87$. GPT-2 small wins, by a wide margin.

In [ ]:
@torch.no_grad()
def total_nll(model, ids, window):
    total = 0.0                               # in nats
    for i in range(0, len(ids) - 1, window):
        x = ids[i:i + window].unsqueeze(0)
        y = ids[i + 1:i + 1 + window].unsqueeze(0)
        x = x[:, :y.shape[1]]
        logits = model(x)
        total += F.cross_entropy(logits[0], y[0],
                                 reduction="sum").item()
    return total

val_text = text[n:]                           # lesson 10.7’s validation text
ids = torch.tensor([stoi[c] for c in val_text])
nats = total_nll(model, ids, 64)
n_bytes = len(val_text[1:].encode("utf-8"))  # predicted text
print(len(ids), n_bytes, round(nats / (len(ids) - 1), 3))
print(round(nats / (math.log(2) * n_bytes), 3))     # bits per byte
# 111540 111539 1.622
# 2.34

The same sum for GPT-2 small, loaded with Hugging Face Transformers. The end-of-text token goes first so that the first real token is predicted too; the tokenizer warns that the text is longer than 1,024 tokens, which is why the loop cuts it into windows:

In [ ]:
from transformers import AutoTokenizer, GPT2LMHeadModel

tok = AutoTokenizer.from_pretrained("gpt2")
gpt2 = GPT2LMHeadModel.from_pretrained("gpt2").eval()
ids = [tok.eos_token_id] + tok.encode(val_text)   # end-of-text first
total = 0.0
with torch.no_grad():
    for i in range(0, len(ids) - 1, 1024):
        chunk = torch.tensor(ids[i:i + 1025])
        logits = gpt2(chunk[None, :-1]).logits[0]
        total += F.cross_entropy(logits, chunk[1:],
                                 reduction="sum").item()
n_tokens = len(ids) - 1
n_bytes = len(val_text.encode("utf-8"))
print(n_tokens, round(total / n_tokens, 3))       # 36059 4.0
print(round(total / (math.log(2) * n_bytes), 3))   # 1.866

### Benchmarks: HellaSwag

A base model does not have to write anything to answer. Score each ending by the model’s average loss on its tokens, given the context, and choose the ending it finds least surprising:

In [ ]:
@torch.no_grad()
def answer(model, tok, ctx, endings):
    ctx_ids = tok.encode(ctx)
    scores = []
    for end in endings:
        end_ids = tok.encode(" " + end)
        x = torch.tensor([ctx_ids + end_ids])
        logits = model(x).logits[0, :-1]
        losses = F.cross_entropy(logits, x[0, 1:], reduction="none")
        scores.append(losses[-len(end_ids):].mean().item())
    return min(range(len(endings)), key=lambda i: scores[i])